# Handling Multiple Sequences

**Colab:** https://colab.research.google.com/github/snjugunanjenga/huggingface-deep-reinforcementlearning-coursework-projects/blob/main/transformers/chapter2/handling_multiple_sequences.ipynb  
**AWS Studio:** https://studiolab.sagemaker.aws/import/github/snjugunanjenga/huggingface-deep-reinforcementlearning-coursework-projects/blob/main/transformers/chapter2/handling_multiple_sequences.ipynb  

YouTube: https://www.youtube.com/watch?v=M6adb1j2jPI

This notebook demonstrates batching, padding, attention masks, and handling long sequences with the `transformers` library (PyTorch backend). Cells contain runnable code; outputs are intentionally cleared for repository storage.

## Setup
Run the following cell in Colab or your environment if you don't have the required packages installed. In Colab you can uncomment and run the install line.

In [ ]:
# Optional: uncomment to install in Colab
# !pip install -q transformers torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

## Models expect a batch of inputs
Transformer models expect a batch dimension (even for a single sequence). Below we show how to build that batch and run a forward pass.

In [ ]:
sequence = "I've been waiting for a HuggingFace course my whole life."

# Manual tokenization (for illustration)
tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)

# Add the batch dimension (models expect batches)
input_ids = torch.tensor([ids])  # shape: (1, seq_len)
print('Input IDs shape:', input_ids.shape)

# Forward pass
outputs = model(input_ids)
print('Logits shape:', outputs.logits.shape)

## Padding the inputs
When batching sequences of different lengths, pad shorter sequences with `tokenizer.pad_token_id`.

In [ ]:
# Example: two sequences of different lengths
seq1 = "I love this!"
seq2 = "This is not good at all."

enc1 = tokenizer(seq1, add_special_tokens=True, return_tensors=None)['input_ids']
enc2 = tokenizer(seq2, add_special_tokens=True, return_tensors=None)['input_ids']

pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
max_len = max(len(enc1), len(enc2))
batched = [enc1 + [pad_id] * (max_len - len(enc1)), enc2 + [pad_id] * (max_len - len(enc2))]
input_ids = torch.tensor(batched)
print('Batched input shape:', input_ids.shape)

# Running without attention mask (may yield different results)
out_no_mask = model(input_ids)
print('Logits without mask shape:', out_no_mask.logits.shape)

## Attention masks
Attention masks tell the model which tokens are padding (0) and which are real tokens (1). Use them when passing padded batches.

In [ ]:
attention_mask = [[1] * len(enc1) + [0] * (max_len - len(enc1)), [1] * len(enc2) + [0] * (max_len - len(enc2))]
attention_mask = torch.tensor(attention_mask)
out_with_mask = model(input_ids, attention_mask=attention_mask)
print('Logits with mask shape:', out_with_mask.logits.shape)

# Compare single-run vs batched+mask (optional verification)
single1 = model(torch.tensor([enc1]))
single2 = model(torch.tensor([enc2]))
print('Single1 logits shape:', single1.logits.shape)
print('Batched logits compare row 1 equal?:', torch.allclose(single1.logits, out_with_mask.logits[0], atol=1e-5))
print('Single2 logits shape:', single2.logits.shape)
print('Batched logits compare row 2 equal?:', torch.allclose(single2.logits, out_with_mask.logits[1], atol=1e-5))

## Longer sequences
Transformers have maximum input lengths (e.g., 512, 1024). To handle longer inputs, either use models designed for long contexts (Longformer, LED) or truncate/segment inputs.

Example: truncate a sequence to `max_length` before tokenization.

In [ ]:
long_seq = "".join(["This is a sentence. " for _ in range(600)])
# Tokenize with truncation
tok = tokenizer(long_seq, truncation=True, max_length=512, return_tensors='pt')
print('Truncated input shape:', tok['input_ids'].shape)
# Use long-context specialized models for bigger windows (see Longformer, LED docs)

---
### Notes & next steps
- This notebook is Colab-ready; the Colab link at the top points to this file in the repository.
- Keep outputs cleared in the repository.
- Consider adding a small `requirements.txt` with tested `transformers`/`torch` versions for reproducibility.